# C1 · Breaking the tree with a trend: flat predictions outside the training range

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Build a series with trend and seasonality, train a linear regression and a regression tree on the first 200 months and predict the next 100: the tree is wrong by almost five times as much as the line, because outside the range of `t` it repeats the last value. Then fix it with a **hybrid** model
(linear trend + tree on the residuals).

## Dataset

`series()` (given): `y = 2 + 0.05 t + 3 sin(2πt/12) + noise` for `t` = 0..299, with features `t`, `sin`, `cos` of the season. Training: `t < 200`; test (future): `t >= 200`.

## Educational purpose

Understand a structural limit: a regression tree predicts **leaf means** of values seen in training, so it cannot produce values beyond the observed range nor continue a trend. It is the problem of tree models (forests and boosting included) with nonstationary series.

## Task

1. Train `LinearRegression` and `DecisionTreeRegressor(min_samples_leaf=5, random_state=0)` on the training set and save `train_rmse` (dictionary with the training error of both) and `future_rmse` (dictionary with the test error, that is in the future).
2. Save `future_tree_predictions`: the **distinct** values predicted by the tree on the future considering only the trend part (use only the feature `t`: train a tree on `t` and predict `t` from 200 to 299): how many are there? Save `n_distinct_values` and `last_constant_value`.
3. **Hybrid:** train the line on the training set, then a tree on the **residuals** (`y − line`) with the same three features and predict the future as `line + tree`. Save `hybrid_rmse`.
4. Save `error_ratio = future_rmse['tree'] / future_rmse['linear']`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor


def series():
    rng = np.random.default_rng(0)
    t = np.arange(300)
    y = 2 + 0.05 * t + 3 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.7, 300)
    return t, np.c_[t, np.sin(2 * np.pi * t / 12), np.cos(2 * np.pi * t / 12)], y


t, F, y = series()
tr, te = t < 200, t >= 200
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))

## Your work

In [ ]:
train_rmse = {}
future_rmse = {}

In [ ]:
n_distinct_values = None
last_constant_value = None

In [ ]:
hybrid_rmse = None
error_ratio = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`lin = LinearRegression().fit(F[tr], y[tr])`; `tree = DecisionTreeRegressor(min_samples_leaf=5, random_state=0).fit(F[tr], y[tr])`. Trend only: `DecisionTreeRegressor(min_samples_leaf=5, random_state=0).fit(t[tr].reshape(-1, 1), y[tr]).predict(t[te].reshape(-1, 1))`; `np.unique(...)`. Hybrid: `res = y[tr] - lin.predict(F[tr]); res_tree = DecisionTreeRegressor(...).fit(F[tr], res)`; prediction `lin.predict(F[te]) + res_tree.predict(F[te])`.'):
    check(set(train_rmse) == {"linear", "tree"} and abs(train_rmse["tree"] - train_rmse["linear"]) < 0.1, "on the training set the two models are on par (within 0.1): nothing hints at the problem", "compare the training errors")
    check(future_rmse["linear"] < 1.0 and future_rmse["tree"] > 3, "in the future the line stays below 1.0; the tree above 3", "test error (t >= 200)")
    check(n_distinct_values is not None and n_distinct_values <= 3 and abs(last_constant_value - y[tr][-30:].mean()) < 3, "a tree on t predicts very few distinct values in the future (at most 3), close to the level of the last months", "the future falls entirely in the last leaf")
    check(hybrid_rmse is not None and hybrid_rmse < 1.0 and hybrid_rmse < future_rmse["tree"] / 3, "the hybrid in the future has RMSE below 1.0 (over 3 times better than the tree)", "line + tree on the residuals")
    check(error_ratio > 4, "the error of the tree is over 4 times that of the line", "future_rmse['tree'] / future_rmse['linear']")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_the_tree_does_not_extrapolate_the_trend.ipynb`. Open it after you pass the check, or after you have really tried.